# Lezione 2 — Backpropagation con il problema XOR

## Obiettivi di apprendimento
- Capire le 5 fasi del backpropagation: forward pass, calcolo dell'errore, backward pass, aggiornamento dei pesi, ripetizione per epoche.
- Implementare il backpropagation con NumPy puro per addestrare una rete che risolva XOR.
- Osservare come l'errore medio diminuisce lungo le epoche di addestramento.
- Capire perché XOR (non essendo linearmente separabile) è l'esempio classico per dimostrare che una rete con strato nascosto impara relazioni non lineari.

## Cosa costruiremo
Addestreremo, da zero, una rete con 2 input, 2 nodi nascosti e 1 output affinché impari la funzione logica XOR. Per far sì che la demo venga eseguita velocemente in lezione, useremo meno epoche e un tasso di apprendimento più alto rispetto al materiale originale (che usa 180.000 epoche con `lr=0.1`) — vedrai lo stesso pattern di convergenza, solo più veloce.

## Ripasso teorico veloce: le 5 fasi del backpropagation

1. **Forward pass**: `X → w1 → sigmoide → a1 → w2 → sigmoide → a2` (la previsione).
2. **Calcolo dell'errore**: `error = d - a2` (valore atteso meno previsione).
3. **Backward pass**: si calcola `dz2` combinando l'errore con la derivata della sigmoide
   nell'output; lo si distribuisce verso lo strato nascosto moltiplicando per `w2.T` per ottenere `da1`,
   e lo si combina di nuovo con la derivata della sigmoide per ottenere `dz1`.
4. **Aggiornamento**: `w2 += lr*dz2·a1.T`, `b2 += lr*sum(dz2)`, `w1 += lr*dz1·X.T`,
   `b1 += lr*sum(dz1)`.
5. **Ripetere** per molte epoche, finché l'errore medio non diventa piccolo.

XOR non può essere separato con una sola linea retta, quindi se la rete impara XOR è perché
sta davvero combinando informazioni in modo non lineare grazie allo strato nascosto.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(1)  # riproducibile in lezione

# tabella di verità di XOR: ogni colonna di X è un esempio (x1, x2)
X = np.array([[0, 0, 1, 1],
              [0, 1, 0, 1]])
d = np.array([[0, 1, 1, 0]])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def initialize_network_parameters():
    inputSize = 2
    hiddenSize = 2
    outputSize = 1
    # NOTA: nel materiale originale si usa lr=0.1 e epochs=180000.
    # Qui usiamo un tasso di apprendimento più alto e meno epoche
    # per far sì che la cella venga eseguita velocemente durante la lezione dal vivo.
    lr = 0.5
    epochs = 20000

    w1 = np.random.uniform(size=(hiddenSize, inputSize), low=-1, high=1)
    b1 = np.random.uniform(size=(hiddenSize, 1), low=-1, high=1)
    w2 = np.random.uniform(size=(outputSize, hiddenSize), low=-1, high=1)
    b2 = np.random.uniform(size=(outputSize, 1), low=-1, high=1)

    return w1, b1, w2, b2, lr, epochs

w1, b1, w2, b2, lr, epochs = initialize_network_parameters()
error_list = []

for epoch in range(epochs):
    # 1. Forward pass
    z1 = np.dot(w1, X) + b1
    a1 = sigmoid(z1)
    z2 = np.dot(w2, a1) + b2
    a2 = sigmoid(z2)

    # 2. Calcolo dell'errore
    error = d - a2

    # 3. Backward pass
    dz2 = error * a2 * (1 - a2)
    dw2 = np.dot(dz2, a1.T)
    db2 = np.sum(dz2, axis=1, keepdims=True)

    da1 = np.dot(w2.T, dz2)
    dz1 = da1 * a1 * (1 - a1)
    dw1 = np.dot(dz1, X.T)
    db1 = np.sum(dz1, axis=1, keepdims=True)

    # 4. Aggiornamento di pesi e bias
    w2 += lr * dw2
    b2 += lr * db2
    w1 += lr * dw1
    b1 += lr * db1

    error_list.append(np.mean(np.abs(error)))

    if epoch % 2000 == 0:
        print('Epoca {}: errore medio = {:.5f}'.format(epoch, np.mean(np.abs(error))))

In [ ]:
# 5. Test finale: la rete ha imparato a risolvere XOR?
z1 = np.dot(w1, X) + b1
a1 = sigmoid(z1)
z2 = np.dot(w2, a1) + b2
a2 = sigmoid(z2)

print('Previsioni finali:', np.round(a2, 5))
print('Valori attesi:     ', d)

plt.plot(error_list)
plt.title('Errore medio lungo le epoche (XOR)')
plt.xlabel('Epoca')
plt.ylabel('Errore medio assoluto')
plt.grid(True)
plt.show()

## Riepilogo della lezione

- Il backpropagation ripete 5 fasi: forward pass, errore, backward pass, aggiornamento, e
  si ricomincia per molte epoche.
- L'errore medio di XOR dovrebbe iniziare vicino a 0.5 (indovinando a caso) e scendere in modo
  costante man mano che avanzano le epoche.
- Abbiamo usato una configurazione di iperparametri diversa da quella del materiale originale
  (`lr=0.5`, `epochs=20000` invece di `lr=0.1`, `epochs=180000`) solo per far sì che la demo sia
  veloce in lezione — il pattern di convergenza è lo stesso, solo con meno precisione
  finale. Nel capitolo del manuale puoi vedere la configurazione originale completa.
- XOR è l'esempio classico perché dimostra che lo strato nascosto permette di imparare una
  relazione che non è linearmente separabile.

**Per approfondire:**
- Capitolo del manuale: [`manual/04_backpropagation.md`](../manual/04_backpropagation_IT.md)
- Notebook di pratica di questa lezione: [`notebooks_practica/clase02_practica_backpropagation.ipynb`](../notebooks_practica/clase02_practica_backpropagation_IT.ipynb)